# Dilute substitutional formation energy

Computes the bare-difference substitutional formation energy of
Ni-in-Cu using the ASE EMT potential. The chemical-potential
correction terms (`mu_solute`, `mu_host`) default to zero, so the
reported value is `E_sub - E_bulk` for an 8x8x8-A supercell.

Driven from a `Workflow` so that `engine.with_working_directory(...)`
is evaluated eagerly with the resolved engine value.


In [1]:
from ase.build import bulk
from ase.calculators.emt import EMT

from pyiron_workflow import Workflow
from pyiron_workflow_atomistics.engine import ASEEngine, CalcInputMinimize, run
from pyiron_workflow_atomistics.structure import (
    create_supercell_with_min_dimensions,
    substitutional_swap,
)
from pyiron_workflow_atomistics.physics.point_defect import (
    _substitutional_formation_energy,
)

engine = ASEEngine(
    EngineInput=CalcInputMinimize(
        force_convergence_tolerance=0.1, max_iterations=50
    ),
    calculator=EMT(),
    working_directory="./_sub_runs",
)


In [2]:
wf = Workflow("substitutional_formation_energy")
wf.supercell = create_supercell_with_min_dimensions(
    bulk("Cu", "fcc", a=3.6, cubic=True), min_dimensions=[8, 8, 8]
)
wf.with_substitute = substitutional_swap(
    wf.supercell, defect_site=0, new_symbol="Ni"
)
wf.calc_bulk = run(
    wf.supercell, engine=engine.with_working_directory("supercell"), label="calc_bulk"
)
wf.calc_sub = run(
    wf.with_substitute, engine=engine.with_working_directory("substitutional"), label="calc_sub"
)
wf.E_form = _substitutional_formation_energy(
    E_sub=wf.calc_sub.outputs.engine_output.final_energy,
    E_bulk=wf.calc_bulk.outputs.engine_output.final_energy,
    mu_solute=0.0,
    mu_host=0.0,
)
wf.run()

e_f = wf.E_form.outputs.E_f.value
print(f"Substitutional formation energy of Ni in Cu (EMT, mu_solute=mu_host=0): "
      f"{e_f:.3f} eV")


      Step     Time          Energy          fmax
BFGS:    0 14:53:08       -0.722387        0.000000


      Step     Time          Energy          fmax
BFGS:    0 14:53:08       -0.638133        0.058053


Substitutional formation energy of Ni in Cu (EMT, mu_solute=mu_host=0): 0.084 eV
